# Module 4: Reusable Parts

**Big idea:** stop copy-pasting the same logic. Name it once as a **function**, call it everywhere, make it survive bad input with **`try/except`**, and save it to a **JSON** file. A small function boundary also lets you swap a model backend without rewriting the assistant.

This notebook runs **completely offline** — no API key, nothing sent to a model. The backend-swap example uses two predictable stand-ins. The functions you build here (`clean_message`, `build_conv`, `safe_clean`, `export_to_json`) are the real ones in `rag-starter/ribot/textprep.py` — the data-prep layer every later week flows through.

## 0. Warm-up — consolidate the two loop shapes

Before any new material: a refresher on Week 3. You learned two loops there.

- **`for`** walks a collection you already have. You wrote these in the lesson and in the
  assignment's `run_chat`.
- **`while`** repeats *until* something changes. You met it gently — as `main()` in
  `ribot/chatbot.py` and as Part 3 of the assignment, the version that reads live `input()`.

Below is the `while` version, side by side with the `for` version of the *same* logic, so the
difference is one line. **Predict what each list holds — and whether the two agree — before
you run it.** The classic bug is forgetting the `.append()`: then the list never grows.

In [ ]:
turns = ["hello", "how are you", "/exit"]   # pretend these are typed one at a time

# --- The `while` shape: repeat UNTIL the user types /exit. --------------------
# This is `main()` in ribot/chatbot.py, with a scripted list standing in for input().
while_history = []
remaining = list(turns)            # a copy we can shrink
while remaining:                   # "keep going while there's anything left"
    msg = remaining.pop(0)         # .pop(0) takes the FIRST item off the list
    if msg == "/exit":
        break                      # the stop condition
    while_history.append({"role": "user", "content": msg})

# --- The `for` shape: do this ONCE PER item. ---------------------------------
for_history = []
for msg in turns:
    if msg == "/exit":
        break
    for_history.append({"role": "user", "content": msg})

print("while_history has", len(while_history), "turns")
print("for_history   has", len(for_history), "turns")
print("same result? ", while_history == for_history)
for turn in for_history:
    print(" ", turn)

## 1. Make the course package importable

The code you build all term lives in the `rag-starter/` repo. Run this once in a terminal from
inside that folder, then select **`rag-starter/.venv`** as the notebook kernel:

```bash
uv venv && uv pip install -e '.[dev]'
```

Because this is an editable install, changes you make to `ribot` are available immediately.
If the package is unavailable, the next cell still uses identical local fallback copies.

In [1]:
# GIVEN — run as-is. Confirm that this notebook kernel can import the course package.
try:
    import ribot
    print("✓ Connected to the editable ribot package.")
except ModuleNotFoundError:
    print("ribot is not installed in this kernel — using local fallback copies below.")

✓ Connected to the editable ribot package.


In [2]:
try:
    from ribot.textprep import clean_message, build_conv, safe_clean, export_to_json
    print("Imported the real functions from ribot.textprep — same code you grow all term.")
except ModuleNotFoundError:
    # Local fallback: identical to ribot/textprep.py so the lesson always runs.
    import json, re

    def clean_message(text):
        """Strip, lowercase, drop punctuation, collapse whitespace."""
        text = text.strip().lower()
        text = re.sub(r"[^\w\s]", "", text)
        text = re.sub(r"\s+", " ", text)
        return text

    def build_conv(questions, answers):
        """Pair questions with answers into cleaned {'user','bot'} dicts."""
        return [
            {"user": clean_message(q), "bot": clean_message(a)}
            for q, a in zip(questions, answers)
        ]

    def safe_clean(text):
        """Clean text but never crash on bad input."""
        try:
            return clean_message(text)
        except (TypeError, AttributeError):
            return ""

    def export_to_json(data, filename):
        """Write any JSON-serializable object to a readable file."""
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

    print("Using local fallback copies (identical to ribot.textprep).")

Imported the real functions from ribot.textprep — same code you grow all term.


## 2. A function — name the logic once

A **function** packages a piece of work behind a name. It has:
- **parameters** — the inputs (here, `text`),
- a **`return`** value — the output handed back,
- a **docstring** — the `"""..."""` line that says what it does.

Instead of pasting the cleaning rule everywhere, you write it **once** and call it by name.

In [ ]:
print(clean_message("  Hello,   WORLD!! "))   # -> 'hello world'
print(clean_message("Where  are the DOCS???"))  # -> 'where are the docs'

# A function's docstring is readable help:
print(clean_message.__doc__)

### Default values — making a parameter optional

A parameter can carry a **default**: `def greet(name, greeting="Hello")`. Callers who don't
pass `greeting` get `"Hello"`; callers who do, override it. Two rules and you know the whole
feature:

1. **Defaults come last.** `def f(a, b=1)` is fine; `def f(a=1, b)` is a syntax error.
2. **The default is evaluated once, when the `def` runs** — so use simple, unchanging values
   (a string, a number, `None`). Never a list or dict as a default.


In [ ]:
def preview(text, length=40, suffix="..."):
    """Shorten text for display. `length` and `suffix` are OPTIONAL — they have defaults."""
    if len(text) <= length:
        return text
    return text[:length] + suffix


long_answer = "Full-time employees receive 15 vacation days per year, accrued monthly."

print(preview(long_answer))                          # both defaults used
print(preview(long_answer, 20))                      # length overridden by position
print(preview(long_answer, length=20, suffix=" >"))  # both overridden BY NAME (clearest)



# ✏️ Your turn 

Give `my_clean` an optional `lowercase=True` parameter, so
`my_clean("HI") -> "hi"` but `my_clean("HI", lowercase=False) -> "HI"`.

## 3. Reuse a function inside another function

`build_conv` pairs a list of questions with a list of answers into a list of `{'user','bot'}` dicts — and it **reuses** `clean_message` so every record is normalized the same way. One cleaning rule, applied everywhere. Fix a bug once, and every caller benefits.

In [3]:
questions = ["  How do I RESET my password? ", "Where are the DOCS?"]
answers   = ["Use the 'Forgot password' link.", "In the help center."]

conv = build_conv(questions, answers)
for record in conv:
    print(record)

{'user': 'how do i reset my password', 'bot': 'use the forgot password link'}
{'user': 'where are the docs', 'bot': 'in the help center'}


### A function boundary makes the model swappable

Week 1 named three backends; now you can see the programming idea that makes them interchangeable. Your assistant should depend on one small contract: **a function receives `messages` and returns a reply string**. The rest of the app should not care where that function runs.

The two tiny backends below are offline stand-ins. Swapping the function changes the answer engine without changing `ask_assistant`. In the course package, `ribot.llm.chat(messages)` is this boundary.

In [ ]:
def api_backend(messages):
    return "API-style reply to: " + messages[-1]["content"]

def local_backend(messages):
    return "Local-style reply to: " + messages[-1]["content"]

def ask_assistant(question, chat_fn):
    messages = [{"role": "user", "content": question}]
    return chat_fn(messages)

question = "Where is the Q3 report?"
print(ask_assistant(question, api_backend))
print(ask_assistant(question, local_backend))

assert ask_assistant(question, api_backend).endswith(question)
assert ask_assistant(question, local_backend).endswith(question)
print("PASS  the caller stayed unchanged; only the backend function changed")

### ✏️ Your turn #1 — turn YOUR cleanup chain into a function

In Week 2 you cleaned a messy question with a chain like `.strip().lower()`. That chain is
still copy-paste — name it **once** as a function. Make it yours: add at least one move your
domain needs (a `.replace(...)` for a common abbreviation, for example) and write a docstring
that says what *your* version does. Then run the self-check.

In [ ]:
def my_clean(text):
    """Tidy a question the way my assistant needs."""      # <- describe YOUR cleanup
    result = text.strip().lower()                          # <- your chain; add e.g. .replace("q3", "third quarter")
    return result

print(my_clean("   WHERE is the Q3 report??  "))           # <- test with a messy question from YOUR domain

In [ ]:
# Self-check (advises, never breaks)
issues = []
if not (my_clean.__doc__ or "").strip():
    issues.append('give my_clean a docstring — the """...""" line right under the def')
elif "the way my assistant needs" in my_clean.__doc__:
    issues.append("the docstring is still the starter text — make it describe YOUR cleanup")
out = my_clean("   TEST Me  ")
if not isinstance(out, str):
    issues.append("my_clean should RETURN a string — did you forget the return?")
else:
    if out != out.strip():
        issues.append("result still has spaces at the ends — is .strip() in your chain?")
    if out != out.lower():
        issues.append("result still has CAPITALS — is .lower() in your chain?")
if not issues:
    print("PASS  my_clean is a real, reusable function ✓")
else:
    for i in issues:
        print("HINT ", i)

### The comprehension twin (a glimpse of the stretch)

`build_conv` is built from a **list comprehension** — a compact way to build a list from another list. The loop version and the comprehension version produce the *same* result; the comprehension is just shorter. (Comprehensions are a stretch goal this week.)

In [ ]:
# Loop version (what a comprehension unrolls into):
loop_version = []
for q, a in zip(questions, answers):
    loop_version.append({"user": clean_message(q), "bot": clean_message(a)})

# Comprehension version (one line — this is what build_conv uses):
comp_version = [{"user": clean_message(q), "bot": clean_message(a)}
                for q, a in zip(questions, answers)]

print("identical?", loop_version == comp_version)

## 4. Reading a list/dict by key and index

The conversation is a **list of dicts**. To read one value you index the list (`[0]`) then the dict by key (`['user']`).

In [ ]:
print("first record:", conv[0])
print("first user message:", conv[0]["user"])
print("last bot message: ", conv[-1]["bot"])

## 5. `try/except`

Real users (and later, the embedding API) sometimes hand you garbage. `clean_message(None)` **crashes** because `None` has no `.strip()`. First, let's *see* the crash and read the **traceback**.

In [ ]:
# Read the traceback below: it names the error TYPE (AttributeError) and the OFFENDING line.
# This cell is EXPECTED to fail — that's the lesson.
try:
    clean_message(None)
except Exception as e:
    print("Crashed with:", type(e).__name__, "->", e)
    print("Lesson: 'NoneType' object has no attribute 'strip' — bad input reached .strip().")

**`try/except`** lets us *try* the risky call and, if it fails, run an *except* branch instead of crashing. `safe_clean` wraps `clean_message` and returns an empty string on bad input. Note we catch **specific** errors (`TypeError`, `AttributeError`) — not a bare `except` that would hide real bugs.

In [ ]:
print(repr(safe_clean("  Hello, WORLD! ")))   # normal input -> 'hello world'
print(repr(safe_clean(None)))                  # bad input  -> '' (no crash)
print(repr(safe_clean(12345)))                 # bad input  -> '' (no crash)
print("The program kept running. That's the point of try/except.")

### ✏️ Your turn #2 — crash-proof a risky call yourself

`safe_clean` wrapped `clean_message`; now make the same move with your own hands.
`safe_first_word` below still crashes two different ways — on `None` (an `AttributeError`)
and on `"   "` (an `IndexError`, because `.split()` gives an empty list). Wrap the risky line
in `try/except` catching those **specific** errors so bad input returns `""` instead.
The cell itself never dies — it shows you each crash until you fix the function.

In [ ]:
def safe_first_word(text):
    """Return the first word of text, or "" if the input is bad."""
    # TODO: put the risky line inside try/except (AttributeError, IndexError)
    #       and return "" in the except branch.
    return text.split()[0]

print(repr(safe_first_word("hello world")))    # 'hello' — already works

# These two CRASH until your try/except is in place (that's the exercise):
for bad in (None, "   "):
    try:
        print(repr(safe_first_word(bad)))
    except Exception as e:
        print(f"CRASHED on {bad!r}: {type(e).__name__} — add the try/except inside safe_first_word")

In [ ]:
# Self-check (advises, never breaks)
issues = []
try:
    if safe_first_word("hello world") != "hello":
        issues.append('safe_first_word("hello world") should return "hello"')
except Exception as e:
    issues.append(f"crashed on GOOD input ({type(e).__name__}) — the try should still return the word")
for bad in (None, "", "   ", 123):
    try:
        if safe_first_word(bad) != "":
            issues.append(f'safe_first_word({bad!r}) should return ""')
    except Exception as e:
        issues.append(f"still crashes on {bad!r} ({type(e).__name__}) — is the risky line inside the try?")
if not issues:
    print("PASS  bad input no longer crashes your function ✓")
else:
    for i in issues:
        print("HINT ", i)

### ✏️ Your card — Week 4: does "no crash" mean "correct"?

Your `try/except` now catches `AttributeError` and `IndexError`. Reasonable question: is the
function *safe* now? Below are four inputs. **Predict each outcome before running the cell** —
write `""`, a value, or the exception name in the Prediction column, then run and compare.

| Input | Prediction | Actual |
|---|---|---|
| `None` | ____ | |
| `"   "` | ____ | |
| `42` | ____ | |
| `b"hi bye"` | ____ | |

- **Hypothesis:** my fixed `safe_first_word` returns `""` for every bad input I can think of.
- **Change:** the *type* of the input (string → `None`, whitespace, `int`, `bytes`).
- **Held fixed:** the function.
- **Metric & prediction:** the four outcomes above, predicted before running.
- **Result & decision:** ____

One of those four does something you probably didn't predict — and it doesn't crash, which is
exactly what makes it interesting. **A test that only asks "did it crash?" cannot catch a
wrong answer.** 


In [ ]:
# Run AFTER you've written your predictions in the table above.
for value in (None, "   ", 42, b"hi bye"):
    try:
        result = safe_first_word(value)
        print(f"{value!r:<12} -> {result!r}   (type: {type(result).__name__})")
    except Exception as e:
        print(f"{value!r:<12} -> raised {type(e).__name__}")

print("\nAny row that surprised you is the most valuable line on your card.")


## 6. Save to JSON — `export_to_json`

**JSON** is the universal "save it to a file" format. `export_to_json` writes any JSON-serializable object (like our `conv`) to a readable `.json` file. The same shape is reused by the document index (Week 6) and the agent's tool results (Week 8).

In [ ]:
export_to_json(conv, "conv_example.json")
print("wrote conv_example.json")

# Read it back to confirm it's a real, readable file:
with open("conv_example.json", encoding="utf-8") as f:
    print(f.read())

### ✏️ Your turn #3 — save YOUR mini-FAQ to JSON

Put the whole lesson together on **your** domain: three real questions your colleagues ask,
three answers, paired by `build_conv` (which reuses the cleaning), saved by `export_to_json`.
This little file is the seed of the document library your bot will search in Week 6.

In [ ]:
my_questions = [
    "  How do I request TIME OFF? ",           # <- three REAL questions from your domain
    "Where is the style guide?",
    "Who approves expense reports?",
]
my_answers = [
    "Use the leave form on the intranet.",     # <- and their answers
    "In the shared drive under /docs.",
    "Your team lead, then finance.",
]

my_faq = build_conv(my_questions, my_answers)
export_to_json(my_faq, "my_faq.json")

with open("my_faq.json", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# Self-check (advises, never breaks)
import json as _json
issues = []
loaded = None
try:
    with open("my_faq.json", encoding="utf-8") as f:
        loaded = _json.load(f)
except FileNotFoundError:
    issues.append("my_faq.json doesn't exist — did the cell above run?")
except _json.JSONDecodeError:
    issues.append("my_faq.json isn't valid JSON — re-run the export cell")
if loaded is not None:
    if len(loaded) < 3:
        issues.append("aim for at least 3 question/answer pairs")
    if not all(isinstance(r, dict) and set(r) == {"user", "bot"} for r in loaded):
        issues.append("each record should have exactly the keys 'user' and 'bot' — build it with build_conv")
    elif any("time off" in r["user"] for r in loaded):
        issues.append("still the example questions — make them YOUR domain's for full value")
if not issues:
    print("PASS  your FAQ round-tripped through JSON ✓")
else:
    for i in issues:
        print("HINT ", i)


## The other direction: the model gives you text, not data

We have sent JSON **out** to a file. This section handles JSON coming **in** from the model.

Here is a more likely situation you will be in. You do not want the model to write you a
paragraph; you want **fields you can use in code**: which category is this question, does it need a
human, what is the ticket number. So you ask for JSON. And the model *mostly* complies.

Typicaly, a model returns a *string*. It
has no obligation to be valid JSON, and three failures are so common they are practically the norm:

| What comes back | Why it breaks |
|---|---|
| `{"topic": "leave", "needs_human": false}` | Nothing — this is the happy path |
| ` ```json ... ``` ` wrapping the object | Markdown fences are text. `json.loads` chokes on the backticks |
| `{"topic": "leave", "needs_hu` | Truncated — it hit a length limit mid-object |

This is **structured output**, and handling it is the difference between a demo and something you can
build on. You already have both tools you need: `try/except` and `json`.

In [ ]:
import json

# Three replies, exactly as models actually return them. Fixtures, not live calls —
# so the failures are reproducible and you can debug them at your own pace.
replies = {
    "clean":     '{"topic": "leave", "needs_human": false}',
    "fenced":    '```json\n{"topic": "leave", "needs_human": false}\n```',
    "truncated": '{"topic": "leave", "needs_hu',
}

# The naive approach: hand each one straight to json.loads and see what happens.
for label, reply in replies.items():
    try:
        parsed = json.loads(reply)
        print(f"{label:<10} OK    -> {parsed}")
    except json.JSONDecodeError as err:
        print(f"{label:<10} CRASH -> {type(err).__name__}: {err}")

Two of the three crash. Note the shape of that output — **`json.loads` raises
`json.JSONDecodeError`**, which is a subclass of `ValueError`, so it is catchable exactly like the
errors.

Now the fix. `parse_model_json` does two things: strip a markdown fence if one is present, then parse
inside a `try/except` and return `None` on failure. **Returning `None` rather than crashing is the
whole point** — your assistant should say "I could not read that" and carry on, not fall over.

In [ ]:
def parse_model_json(reply):
    """Turn a model's reply string into a dict — or None if it isn't usable JSON.

    Handles the two common failures: a ```json markdown fence, and invalid/truncated JSON.
    """
    text = reply.strip()

    # 1) Strip a markdown fence if the model added one.
    if text.startswith("```"):
        text = text.split("\n", 1)[-1]        # drop the opening ```json line
        text = text.removesuffix("```").strip()  # drop the closing fence

    # 2) Parse, but never crash the caller.
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


for label, reply in replies.items():
    result = parse_model_json(reply)
    verdict = "parsed" if result is not None else "unreadable (None)"
    print(f"{label:<10} {verdict:<18} -> {result}")

Two parse, one honestly reports failure. That is the correct outcome — a truncated object
*should* come back as `None`, because guessing what the missing half said would be worse than
admitting you cannot read it.

> **`.removesuffix()`** is a string method like the ones from Week 2 — it removes the ending if it is
> there and does nothing if it is not, so no `if` is needed around it.

### ✏️ Your turn #4 — the full pipeline, both directions

Chain the two halves of this week together: **model text → dict → file.** Parse a reply with
`parse_model_json`, then save the result with your `export_to_json`. Skip the replies that
came back `None` — never write an unreadable record into your data.

In [ ]:
# A batch of replies, as if three questions had been classified by the model.
# One of them is deliberately broken — your pipeline has to survive it.
batch = [
    '{"question": "how do I request time off?", "topic": "leave", "needs_human": false}',
    '```json\n{"question": "who approves expenses?", "topic": "finance", "needs_human": true}\n```',
    '{"question": "where is the style guide?", "topic": "do',   # truncated
]

good = []
skipped = 0

for reply in batch:
    record = parse_model_json(reply)      # <- TODO: nothing to change, but read this line
    if record is None:
        skipped += 1                      # <- TODO: what should happen to an unreadable reply?
        continue
    good.append(record)

export_to_json(good, "classified.json")

print(f"parsed  : {len(good)}")
print(f"skipped : {skipped}")
print()
with open("classified.json", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# Self-check (advises, never breaks)
issues = []

if parse_model_json('{"a": 1}') != {"a": 1}:
    issues.append("plain JSON should parse to a dict")
if parse_model_json('```json\n{"a": 1}\n```') != {"a": 1}:
    issues.append("a fenced reply should parse too — strip the ``` lines first")
if parse_model_json('{"a": ') is not None:
    issues.append("broken JSON must return None, not crash and not a guess")
if parse_model_json("not json at all") is not None:
    issues.append("plain prose isn't JSON — that should be None as well")

try:
    with open("classified.json", encoding="utf-8") as f:
        saved = json.load(f)
    if len(saved) != 2:
        issues.append(f"expected 2 good records in classified.json, found {len(saved)}")
except FileNotFoundError:
    issues.append("classified.json missing — run the pipeline cell above")

if not issues:
    print("PASS  you can take a model's text and get data out of it ✓")
else:
    for i in issues:
        print("HINT ", i)

### Where this goes

- **Week 8** asks the model to pick a *tool* by name. That is the same problem wearing a different
  hat: the model must produce a structure your code can act on, and your code must survive it not
  doing so. You will recognise the shape.
- **Week 10** hardens it — a `None` from `parse_model_json` is exactly the moment a grounded
  assistant should refuse rather than improvise.

> **Want to see it live, offline?** `ribot`'s offline model plays along. Put the marker in the system
> prompt and `FakeLLM` answers with JSON — fenced, because that is what real models do:
>
> ```py
> from ribot.llm import chat, JSON_PROTOCOL_MARKER
>
> messages = [
>     {"role": "system", "content": f"Extract the fields. {JSON_PROTOCOL_MARKER}."},
>     {"role": "user", "content": "how many vacation days do I get?"},
> ]
> print(parse_model_json(chat(messages)))     # -> a dict, fence and all handled
> ```
>
> No API key needed. After Key Day in Week 5, the same code runs against a real model and the
> `try/except` starts earning its keep.

## 7. Stretch — recursion: `build_MT_conv`

> **(stretch — skip freely)** Nothing later in the course depends on this section.

A **multi-turn** dialogue can nest: a `bot` value might itself be a list of more turns. **Recursion** is a function that calls *itself* to handle each nested level. Trace the call stack on paper before reading the code: the function flattens nested turns into one flat list of cleaned records.

*(Stretch — you write your own version in the assignment.)*

In [ ]:
def build_MT_conv(dialogue):
    """Flatten a possibly-nested multi-turn dialogue into one flat list of clean records."""
    flat = []
    for turn in dialogue:
        bot = turn["bot"]
        if isinstance(bot, list):
            # nested turns -> recurse into them
            flat.extend(build_MT_conv(bot))
        else:
            flat.append({"user": clean_message(turn["user"]),
                         "bot": clean_message(bot)})
    return flat

nested = [
    {"user": "hello", "bot": "hi"},
    {"user": "how are you?", "bot": [
        {"user": "how do you work?", "bot": "I run on Python code"},
        {"user": "are you trained?", "bot": "Yes, using ML models"},
    ]},
]
for record in build_MT_conv(nested):
    print(record)

## Recap & what's next

You turned copy-pasted logic into **reusable functions**, made them crash-proof with **`try/except`**, and saved structured data to **JSON**.

- **Two loop shapes, consolidated** — `for` walks a collection; `while` repeats until
  something changes. Same body, different "go round again" line.
- **Functions** (`clean_message`, `build_conv`) — name the logic once, reuse it.
- **Default parameter values** (`def preview(text, length=40)`) — optional inputs. You'll
  write these in Week 7 and read them in every week between.
- **`try/except`** (`safe_clean`) — bad input returns a safe value, not a crash. **(floor)**
- **Reading a traceback** — find the error type and the offending line.
- **JSON** (`export_to_json`) — save a readable file the rest of the app depends on. **(floor)**
- **Structured output** (`parse_model_json`) — JSON coming *in* from a model: strip the fence,
  parse in a `try/except`, return `None` rather than crash or guess. **(floor)** Week 8's tool
  schema is this same problem, and Week 10's guardrail is what a `None` should trigger.
- **Stretch:** recursion (`build_MT_conv`) and comprehensions.

**Next (Week 5):** these function boundaries are exactly where the embedding call plugs in — and `safe_clean` is the pattern that keeps it alive on bad vectors. On to the assignment: write your own versions and pass the self-check.